In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.decomposition import PCA
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import pennylane as qml
from pennylane import numpy as pnp

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [2]:
train_df = pd.read_csv("../data/processed/train_processed.csv")
test_df = pd.read_csv("../data/processed/test_processed.csv")

X_train = train_df.drop(columns=["Yield"])
y_train = train_df["Yield"]

X_test = test_df.drop(columns=["Yield"])
y_test = test_df["Yield"]

print(X_train.shape)
print(X_test.shape)

(6513, 19)
(1050, 19)


In [3]:
pca = PCA(n_components=4)

X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print(X_train_pca.shape)

(6513, 4)


In [4]:
n_qubits = 4

dev = qml.device(
    "default.qubit",
    wires=n_qubits
)

In [10]:
@qml.qnode(dev)
def quantum_feature_map(x):

    qml.AngleEmbedding(x, wires=range(n_qubits))

    qml.BasicEntanglerLayers(
        weights=np.array([[0.1, 0.2, 0.3, 0.4]]),
        wires=range(n_qubits)
    )

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

In [11]:
X_train_quantum = np.array(
    [quantum_feature_map(x) for x in X_train_pca]
)

X_test_quantum = np.array(
    [quantum_feature_map(x) for x in X_test_pca]
)

print(X_train_quantum.shape)
print(X_test_quantum.shape)

(6513, 4)
(1050, 4)


In [12]:
from sklearn.ensemble import RandomForestRegressor

rf_quantum = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_quantum.fit(X_train_quantum, y_train)

print("Quantum Random Forest trained!")

Quantum Random Forest trained!


In [13]:
y_pred_quantum = rf_quantum.predict(X_test_quantum)

In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae_q = mean_absolute_error(y_test, y_pred_quantum)

rmse_q = np.sqrt(mean_squared_error(y_test, y_pred_quantum))

r2_q = r2_score(y_test, y_pred_quantum)

print("Quantum MAE :", mae_q)
print("Quantum RMSE:", rmse_q)
print("Quantum R² :", r2_q)

Quantum MAE : 2.195334238095238
Quantum RMSE: 3.360156228462273
Quantum R² : -0.04242410207096747


In [16]:
comparison = pd.DataFrame({
    "Model": [
        "Random Forest",
        "XGBoost",
        "Hybrid Quantum RF"
    ],
    "MAE": [
        mae,
        mae_xgb,
        mae_q
    ],
    "RMSE": [
        rmse,
        rmse_xgb,
        rmse_q
    ],
    "R2": [
        r2,
        r2_xgb,
        r2_q
    ]
})

comparison

NameError: name 'mae' is not defined

In [17]:
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rf = joblib.load("../models/random_forest_model.pkl")
xgb = joblib.load("../models/xgboost_model.pkl")

In [18]:
rf_pred = rf.predict(X_test)

mae = mean_absolute_error(y_test, rf_pred)
rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
r2 = r2_score(y_test, rf_pred)

In [19]:
xgb_pred = xgb.predict(X_test)

mae_xgb = mean_absolute_error(y_test, xgb_pred)
rmse_xgb = np.sqrt(mean_squared_error(y_test, xgb_pred))
r2_xgb = r2_score(y_test, xgb_pred)

In [20]:
comparison = pd.DataFrame({
    "Model": [
        "Random Forest",
        "XGBoost",
        "Hybrid Quantum RF"
    ],
    "MAE": [
        mae,
        mae_xgb,
        mae_q
    ],
    "RMSE": [
        rmse,
        rmse_xgb,
        rmse_q
    ],
    "R2 Score": [
        r2,
        r2_xgb,
        r2_q
    ]
})

comparison

,Model,MAE,RMSE,R2 Score
0,Random Forest,0.350734,0.700260,0.954726
1,XGBoost,0.426647,0.874159,0.929448
2,Hybrid Quantum RF,2.195334,3.360156,-0.042424
